## News Scrape Data Processing Pipeline

In [1]:
from pathlib import Path
import json
from datetime import datetime, UTC
import re

import numpy as np
import pandas as pd

PREFERRED_BASE_DIR = Path("/home/billy/X/Election-Dashboard")


def find_project_root() -> Path:
    expected_rel = Path("data_science/data/raw/news_scrape")

    if (PREFERRED_BASE_DIR / expected_rel).exists():
        return PREFERRED_BASE_DIR

    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / expected_rel).exists():
            return candidate

    return PREFERRED_BASE_DIR


BASE_DIR = find_project_root()
RAW_DIR = BASE_DIR / "data_science" / "data" / "raw" / "news_scrape"
PROCESSED_DIR = BASE_DIR / "data_science" / "data" / "processed" / "news_scrape"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RAW_DIR, PROCESSED_DIR

STANDARD_COLUMNS = [
    "source", "source_file", "dataset_type", "url", "page_title", "scraped_at",
    "table_index", "row_index", "constituency", "candidates", "party",
    "votes", "turnout", "margin", "referendum_yes", "referendum_no",
    "male_voter", "female_voter",
]

REFERENDUM_COLUMNS = [
    "source", "source_file", "dataset_type", "seat_id", "seat_no", "constituency",
    "division_id", "turnout", "referendum_yes", "referendum_no", "male_voter",
    "female_voter", "total_center", "seats_area", "result_id",
]

COLUMNS_TO_REMOVE_FROM_OUTPUT = [
    "source", "source_file", "dataset_type", "url", "page_title", "scraped_at",
    "table_index", "row_index",
]
OUTPUT_COLUMNS = [col for col in STANDARD_COLUMNS if col not in COLUMNS_TO_REMOVE_FROM_OUTPUT]

CANDIDATE_HINTS = {"candidates", "candidate", "candidate_name", "party", "votes"}
REFERENDUM_HINTS = {"seat_name", "yes", "no", "male_voter", "female_voter", "total_voter"}
DHAKA_TRIBUNE_CANDIDATE_HINTS = {"seat_id", "requested_seat_id", "name", "league", "vote"}


def detect_dataset_type(path: Path, df: pd.DataFrame | None = None) -> str:
    name = path.stem.lower()
    if "referendum" in name or "seat" in name:
        return "referendum"
    if df is not None:
        columns = {str(column).strip().lower() for column in df.columns}
        if columns & REFERENDUM_HINTS:
            return "referendum"
        if columns & CANDIDATE_HINTS:
            return "candidate"
    for key in ["raw", "standard", "candidates"]:
        if f"_{key}_" in name or name.endswith(f"_{key}"):
            return key
    return "unknown"


def load_file(path: Path) -> pd.DataFrame:
    if path.suffix.lower() == ".csv":
        df = pd.read_csv(path)
    elif path.suffix.lower() == ".json":
        with path.open("r", encoding="utf-8") as f:
            payload = json.load(f)

        if isinstance(payload, list):
            if len(payload) == 0:
                return pd.DataFrame(columns=STANDARD_COLUMNS)
            df = pd.json_normalize(payload)
        elif isinstance(payload, dict):
            if "data" in payload and isinstance(payload["data"], list):
                df = pd.json_normalize(payload["data"])
            else:
                df = pd.json_normalize([payload])
        else:
            return pd.DataFrame(columns=STANDARD_COLUMNS)
    else:
        return pd.DataFrame(columns=STANDARD_COLUMNS)

    return df


def clean_numeric(series: pd.Series) -> pd.Series:
    cleaned = (
        series.astype(str)
        .str.replace(",", "", regex=False)
        .str.replace(r"[^\d\.-]", "", regex=True)
        .replace({"": np.nan, "nan": np.nan, "None": np.nan})
    )
    return pd.to_numeric(cleaned, errors="coerce")


DISTRICT_ALIAS_MAP = {
    "barisal": "Barishal",
    "barishal": "Barishal",
    "bogra": "Bogura",
    "bogura": "Bogura",
    "chattogram": "Chattogram",
    "chittagong": "Chattogram",
    "chapainababganj": "Chapainababganj",
    "chapainawabganj": "Chapainababganj",
    "comilla": "Cumilla",
    "cumilla": "Cumilla",
    "coxs bazar": "Cox's Bazar",
    "jashore": "Jashore",
    "jessore": "Jashore",
    "jhalakathi": "Jhalokathi",
    "jhalokathi": "Jhalokathi",
    "khagrachari": "Khagrachhari",
    "khagrachhari": "Khagrachhari",
    "kishoreganj": "Kishoreganj",
    "maulvibazar": "Moulvibazar",
    "moulvibazar": "Moulvibazar",
    "netrakona": "Netrakona",
    "netrokona": "Netrakona",
}

PARTY_ALIAS_MAP = {
    "bnp": "BNP",
    "bangladesh nationalist party bnp": "BNP",
    "islami andolon bangladesh": "Islami Andolon Bangladesh",
    "islamic andolon bangladesh": "Islami Andolon Bangladesh",
    "jatiya party": "Jatiya Party (JaPa)",
    "jatiya party japa": "Jatiya Party (JaPa)",
    "gono odhikar parishad": "Gono Odhikar Parishad (GOP)",
    "gono odhikar parishad gop": "Gono Odhikar Parishad (GOP)",
    "national citizen party ncp": "National Citizen Party (NCP)",
    "national citizens party ncp": "National Citizen Party (NCP)",
}

SINGLE_SEAT_DISTRICT_MAP = {
    "Bandarban": "Bandarban 1",
    "Khagrachhari": "Khagrachhari 1",
    "Rangamati": "Rangamati 1",
}


def canonicalize_district_name(name: str) -> str:
    key = (
        str(name)
        .lower()
        .replace("\u2019", "'")
        .replace("`", "'")
        .replace("'", "")
    )
    key = re.sub(r"[^a-z0-9]+", " ", key)
    key = re.sub(r"\s+", " ", key).strip()
    return DISTRICT_ALIAS_MAP.get(key, str(name))


def normalize_constituency_name(value: object) -> str:
    if pd.isna(value):
        return ""

    normalized = re.sub(r"\s+", " ", str(value).replace("-", " ")).strip()
    if normalized == "":
        return ""

    match = re.match(r"^(.*?)(\s+\d+)?$", normalized)
    district = (match.group(1) if match else normalized).strip()
    seat_suffix = (match.group(2) if match else "") or ""

    canonical_district = canonicalize_district_name(district)
    canonical_constituency = f"{canonical_district}{seat_suffix}"
    return SINGLE_SEAT_DISTRICT_MAP.get(canonical_constituency, canonical_constituency)


def normalize_party_name(value: object) -> str:
    if pd.isna(value):
        return ""

    normalized = re.sub(r"\s+", " ", str(value)).strip()
    if normalized == "":
        return ""

    key = (
        normalized.lower()
        .replace("\u2019", "'")
        .replace("`", "'")
        .replace("'", "")
    )
    key = re.sub(r"[^a-z0-9]+", " ", key)
    key = re.sub(r"\s+", " ", key).strip()
    return PARTY_ALIAS_MAP.get(key, normalized)


def first_non_null(values: pd.Series) -> object:
    for value in values:
        if pd.notna(value) and str(value).strip() != "":
            return value
    return pd.NA


def count_non_null(df: pd.DataFrame, column: str) -> int:
    if column not in df.columns or df.empty:
        return 0
    return int(df[column].notna().sum())


def standardize_candidate_rows(df: pd.DataFrame, file_path: Path) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame(columns=STANDARD_COLUMNS)

    rename_map = {
        "candidate": "candidates",
        "candidate_name": "candidates",
        "candidate_names": "candidates",
        "constituency_name": "constituency",
        "seat": "constituency",
    }
    df = df.rename(columns=rename_map)

    for col in STANDARD_COLUMNS:
        if col not in df.columns:
            df[col] = pd.NA

    if df["source"].isna().all() or (df["source"].astype(str).str.strip() == "").all():
        df["source"] = file_path.stem.split("_")[0]

    df["source_file"] = file_path.name
    df["dataset_type"] = detect_dataset_type(file_path, df)
    df["constituency"] = df["constituency"].map(normalize_constituency_name)
    df["party"] = df["party"].map(normalize_party_name)

    text_cols = ["source", "source_file", "dataset_type", "url", "page_title", "constituency", "candidates", "party"]
    for col in text_cols:
        df[col] = (
            df[col]
            .astype("string")
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
            .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
        )

    for col in ["votes", "turnout", "margin", "row_index", "table_index", "referendum_yes", "referendum_no", "male_voter", "female_voter"]:
        df[col] = clean_numeric(df[col])

    df["scraped_at"] = pd.to_datetime(df["scraped_at"], errors="coerce", utc=True)

    out = df[STANDARD_COLUMNS].copy()
    out = out.drop_duplicates()
    return out


def standardize_dhaka_tribune_candidate_rows(
    df: pd.DataFrame,
    file_path: Path,
    seat_lookup_by_id: dict[int, str],
) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame(columns=STANDARD_COLUMNS)

    rename_map = {
        "name": "candidates",
        "league": "party",
        "vote": "votes",
    }
    df = df.rename(columns=rename_map)

    for col in STANDARD_COLUMNS:
        if col not in df.columns:
            df[col] = pd.NA
    for col in ["seat_id", "requested_seat_id"]:
        if col not in df.columns:
            df[col] = pd.NA

    if df["source"].isna().all() or (df["source"].astype(str).str.strip() == "").all():
        df["source"] = file_path.stem.split("_")[0]

    df["source_file"] = file_path.name
    df["dataset_type"] = "candidate"
    df["seat_id"] = clean_numeric(df["seat_id"])
    df["requested_seat_id"] = clean_numeric(df["requested_seat_id"])
    df["constituency"] = df["seat_id"].map(lambda value: seat_lookup_by_id.get(int(value)) if pd.notna(value) else pd.NA)
    missing_constituency = df["constituency"].isna() & df["requested_seat_id"].notna()
    df.loc[missing_constituency, "constituency"] = df.loc[missing_constituency, "requested_seat_id"].map(
        lambda value: seat_lookup_by_id.get(int(value)) if pd.notna(value) else pd.NA
    )
    df["constituency"] = df["constituency"].map(normalize_constituency_name)
    df["party"] = df["party"].map(normalize_party_name)

    text_cols = ["source", "source_file", "dataset_type", "url", "page_title", "constituency", "candidates", "party"]
    for col in text_cols:
        df[col] = (
            df[col]
            .astype("string")
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
            .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
        )

    for col in ["votes", "turnout", "margin", "row_index", "table_index", "referendum_yes", "referendum_no", "male_voter", "female_voter", "seat_id", "requested_seat_id"]:
        df[col] = clean_numeric(df[col])

    df["scraped_at"] = pd.to_datetime(df["scraped_at"], errors="coerce", utc=True)

    out = df[STANDARD_COLUMNS].copy()
    out = out.drop_duplicates()
    return out


def standardize_referendum_rows(df: pd.DataFrame, file_path: Path) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame(columns=REFERENDUM_COLUMNS)

    rename_map = {
        "id": "seat_id",
        "seat_name": "constituency",
        "yes": "referendum_yes",
        "no": "referendum_no",
        "total_voter": "turnout",
    }
    df = df.rename(columns=rename_map)

    for col in REFERENDUM_COLUMNS:
        if col not in df.columns:
            df[col] = pd.NA

    if df["source"].isna().all() or (df["source"].astype(str).str.strip() == "").all():
        df["source"] = file_path.stem.split("_")[0]

    df["source_file"] = file_path.name
    df["dataset_type"] = "referendum"
    df["constituency"] = df["constituency"].map(normalize_constituency_name)

    text_cols = ["source", "source_file", "dataset_type", "constituency", "seats_area"]
    for col in text_cols:
        df[col] = (
            df[col]
            .astype("string")
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
            .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
        )

    for col in [
        "seat_id", "seat_no", "division_id", "turnout", "referendum_yes", "referendum_no",
        "male_voter", "female_voter", "total_center", "result_id",
    ]:
        df[col] = clean_numeric(df[col])

    out = df[REFERENDUM_COLUMNS].copy()
    out = out.drop_duplicates()
    return out

In [2]:
dhaka_tribune_candidate_files = sorted(RAW_DIR.glob("dhaka_tribune_candidates_raw_*.csv"))
if not dhaka_tribune_candidate_files:
    raise FileNotFoundError("No Dhaka Tribune candidate CSV found matching 'dhaka_tribune_candidates_raw_*.csv'")
dhaka_tribune_candidate_file = dhaka_tribune_candidate_files[-1]

dhaka_tribune_referendum_files = sorted(RAW_DIR.glob("dhaka_tribune_referendum_raw_*.csv"))
dhaka_tribune_seat_files = sorted(RAW_DIR.glob("dhaka_tribune_seats_raw_*.csv"))
daily_star_candidate_files = sorted(RAW_DIR.glob("daily_star_candidates_*.csv"))
daily_star_candidate_file = daily_star_candidate_files[-1] if daily_star_candidate_files else None

dhaka_tribune_referendum_source_files: list[Path] = []
if dhaka_tribune_referendum_files:
    dhaka_tribune_referendum_source_files.append(dhaka_tribune_referendum_files[-1])
if dhaka_tribune_seat_files:
    dhaka_tribune_referendum_source_files.append(dhaka_tribune_seat_files[-1])

if not dhaka_tribune_referendum_source_files:
    raise FileNotFoundError(
        "No Dhaka Tribune referendum CSV found. Expected one of "
        "'dhaka_tribune_referendum_raw_*.csv' or 'dhaka_tribune_seats_raw_*.csv'"
    )

print(f"Using Dhaka Tribune candidate file: {dhaka_tribune_candidate_file.name}")
print("Using Dhaka Tribune referendum files:", ", ".join(path.name for path in dhaka_tribune_referendum_source_files))
if daily_star_candidate_file is not None:
    print(f"Using Daily Star candidate file: {daily_star_candidate_file.name}")
else:
    print("No Daily Star candidate file found; alliance enrichment will be skipped.")

candidate_frames = []
referendum_frames = []
file_level_stats = []

for path in dhaka_tribune_referendum_source_files:
    try:
        raw_df = load_file(path)
        clean_df = standardize_referendum_rows(raw_df, path)
        referendum_frames.append(clean_df)

        file_level_stats.append(
            {
                "source_file": path.name,
                "dataset_type": "referendum",
                "rows_in_raw": int(len(raw_df)),
                "rows_after_cleaning": int(len(clean_df)),
                "non_null_constituency": count_non_null(clean_df, "constituency"),
                "non_null_candidates": count_non_null(clean_df, "candidates"),
                "non_null_referendum_yes": count_non_null(clean_df, "referendum_yes"),
                "non_null_male_voter": count_non_null(clean_df, "male_voter"),
            }
        )
    except Exception as exc:
        file_level_stats.append(
            {
                "source_file": path.name,
                "dataset_type": "referendum",
                "rows_in_raw": 0,
                "rows_after_cleaning": 0,
                "non_null_constituency": 0,
                "non_null_candidates": 0,
                "non_null_referendum_yes": 0,
                "non_null_male_voter": 0,
                "error": str(exc),
            }
        )

referendum_df = pd.concat(referendum_frames, ignore_index=True) if referendum_frames else pd.DataFrame(columns=REFERENDUM_COLUMNS)
if referendum_df.empty:
    raise ValueError("No Dhaka Tribune referendum/seat rows were loaded.")

seat_lookup_df = referendum_df[["seat_id", "constituency"]].dropna(subset=["seat_id", "constituency"]).copy()
if seat_lookup_df.empty:
    raise ValueError("Unable to build seat lookup from Dhaka Tribune referendum/seat files.")

seat_lookup_df["seat_lookup_non_null_count"] = seat_lookup_df[["constituency"]].notna().sum(axis=1)
seat_lookup_df = seat_lookup_df.sort_values(["seat_id", "seat_lookup_non_null_count"], ascending=[True, False])
seat_lookup_df = seat_lookup_df.groupby("seat_id", as_index=False).agg({"constituency": first_non_null})
seat_lookup_by_id = {
    int(row["seat_id"]): normalize_constituency_name(row["constituency"])
    for _, row in seat_lookup_df.iterrows()
    if pd.notna(row["seat_id"]) and pd.notna(row["constituency"])
}

try:
    raw_df = load_file(dhaka_tribune_candidate_file)
    clean_df = standardize_dhaka_tribune_candidate_rows(raw_df, dhaka_tribune_candidate_file, seat_lookup_by_id)
    candidate_frames.append(clean_df)

    file_level_stats.append(
        {
            "source_file": dhaka_tribune_candidate_file.name,
            "dataset_type": "candidate",
            "rows_in_raw": int(len(raw_df)),
            "rows_after_cleaning": int(len(clean_df)),
            "non_null_constituency": count_non_null(clean_df, "constituency"),
            "non_null_candidates": count_non_null(clean_df, "candidates"),
            "non_null_referendum_yes": count_non_null(clean_df, "referendum_yes"),
            "non_null_male_voter": count_non_null(clean_df, "male_voter"),
        }
    )
except Exception as exc:
    file_level_stats.append(
        {
            "source_file": dhaka_tribune_candidate_file.name,
            "dataset_type": "candidate",
            "rows_in_raw": 0,
            "rows_after_cleaning": 0,
            "non_null_constituency": 0,
            "non_null_candidates": 0,
            "non_null_referendum_yes": 0,
            "non_null_male_voter": 0,
            "error": str(exc),
        }
    )

candidate_df = pd.concat(candidate_frames, ignore_index=True) if candidate_frames else pd.DataFrame(columns=STANDARD_COLUMNS)
if candidate_df.empty:
    raise ValueError("No Dhaka Tribune candidate rows were loaded.")

referendum_lookup_columns = ["turnout", "referendum_yes", "referendum_no", "male_voter", "female_voter"]
referendum_df = referendum_df.copy()
referendum_df["constituency"] = referendum_df["constituency"].map(normalize_constituency_name)
referendum_df["lookup_non_null_count"] = referendum_df[referendum_lookup_columns].notna().sum(axis=1)
referendum_df = referendum_df.sort_values(["constituency", "lookup_non_null_count"], ascending=[True, False])
referendum_lookup = referendum_df.groupby("constituency", as_index=False).agg(
    {
        "turnout": first_non_null,
        "referendum_yes": first_non_null,
        "referendum_no": first_non_null,
        "male_voter": first_non_null,
        "female_voter": first_non_null,
    }
)

processed_news = candidate_df.copy()
processed_news["constituency"] = processed_news["constituency"].map(normalize_constituency_name)

processed_news = processed_news.merge(referendum_lookup, on="constituency", how="left", suffixes=("", "_referendum"))
for col in referendum_lookup_columns:
    referendum_col = f"{col}_referendum"
    if referendum_col in processed_news.columns:
        processed_news[col] = processed_news[col].combine_first(processed_news[referendum_col])
        processed_news = processed_news.drop(columns=[referendum_col])

if "votes" in processed_news.columns:
    votes_by_constituency = (
        processed_news.groupby("constituency", dropna=True)["votes"]
        .apply(lambda values: [float(v) for v in values.dropna().sort_values(ascending=False).tolist()])
    )
    margin_by_constituency = {}
    for constituency, ranked_votes in votes_by_constituency.items():
        if len(ranked_votes) >= 2:
            margin_by_constituency[constituency] = ranked_votes[0] - ranked_votes[1]
        elif len(ranked_votes) == 1:
            margin_by_constituency[constituency] = ranked_votes[0]
        else:
            margin_by_constituency[constituency] = pd.NA

    processed_news["margin"] = processed_news["margin"].combine_first(
        processed_news["constituency"].map(margin_by_constituency)
    )

processed_news["party"] = processed_news["party"].map(normalize_party_name)

alliance_lookup = pd.DataFrame(columns=["constituency", "alliance"])
if daily_star_candidate_file is not None:
    try:
        daily_star_raw_df = load_file(daily_star_candidate_file)
        daily_star_df = daily_star_raw_df.copy()

        if {"constituency", "alliance"}.issubset(daily_star_df.columns):
            daily_star_df["constituency"] = daily_star_df["constituency"].map(normalize_constituency_name)
            daily_star_df["alliance"] = (
                daily_star_df["alliance"]
                .astype("string")
                .str.replace(r"\s+", " ", regex=True)
                .str.strip()
                .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
            )
            alliance_lookup = daily_star_df.groupby("constituency", as_index=False).agg({"alliance": first_non_null})
        else:
            print("Daily Star file is missing constituency/alliance columns; skipping alliance enrichment.")

        file_level_stats.append(
            {
                "source_file": daily_star_candidate_file.name,
                "dataset_type": "daily_star_candidate",
                "rows_in_raw": int(len(daily_star_raw_df)),
                "rows_after_cleaning": int(len(alliance_lookup)),
                "non_null_constituency": count_non_null(alliance_lookup, "constituency"),
                "non_null_candidates": 0,
                "non_null_referendum_yes": 0,
                "non_null_male_voter": 0,
            }
        )
    except Exception as exc:
        file_level_stats.append(
            {
                "source_file": daily_star_candidate_file.name,
                "dataset_type": "daily_star_candidate",
                "rows_in_raw": 0,
                "rows_after_cleaning": 0,
                "non_null_constituency": 0,
                "non_null_candidates": 0,
                "non_null_referendum_yes": 0,
                "non_null_male_voter": 0,
                "error": str(exc),
            }
        )

if not alliance_lookup.empty:
    processed_news = processed_news.merge(alliance_lookup, on="constituency", how="left", suffixes=("", "_daily_star"))
    if "alliance_daily_star" in processed_news.columns:
        if "alliance" in processed_news.columns:
            processed_news["alliance"] = processed_news["alliance"].combine_first(processed_news["alliance_daily_star"])
        else:
            processed_news["alliance"] = processed_news["alliance_daily_star"]
        processed_news = processed_news.drop(columns=["alliance_daily_star"])
processed_news = processed_news.drop_duplicates()

reference_constituencies = set(referendum_lookup["constituency"].dropna().astype(str))
candidate_constituencies = set(processed_news["constituency"].dropna().astype(str))
missing_constituencies = sorted(reference_constituencies - candidate_constituencies)
if missing_constituencies:
    raise ValueError(
        "Candidate rows are missing constituencies from Dhaka Tribune seat universe: "
        f"{missing_constituencies}"
    )

unique_constituencies = int(processed_news["constituency"].nunique(dropna=True))
if unique_constituencies != 300:
    raise ValueError(f"Expected 300 distinct constituencies, found {unique_constituencies}")

output_columns = [
    "constituency",
    "candidates",
    "party",
    "alliance",
    "votes",
    "turnout",
    "margin",
    "referendum_yes",
    "referendum_no",
    "male_voter",
    "female_voter",
]
processed_news = processed_news[[column for column in output_columns if column in processed_news.columns]]
processed_news = processed_news.sort_values(["constituency", "votes"], ascending=[True, False], na_position="last").reset_index(drop=True)

run_stamp = datetime.now(UTC).strftime("%Y%m%d_%H%M%S")
news_out_csv = PROCESSED_DIR / "news_scrape_processed.csv"
news_out_json = PROCESSED_DIR / "news_scrape_processed.json"
news_out_csv_versioned = PROCESSED_DIR / f"news_scrape_processed_{run_stamp}.csv"
news_out_json_versioned = PROCESSED_DIR / f"news_scrape_processed_{run_stamp}.json"
stats_out_csv = PROCESSED_DIR / "news_scrape_file_stats.csv"

processed_news.to_csv(news_out_csv, index=False)
processed_news.to_csv(news_out_csv_versioned, index=False)
processed_news.to_json(news_out_json, orient="records", force_ascii=False, indent=2, date_format="iso")
processed_news.to_json(news_out_json_versioned, orient="records", force_ascii=False, indent=2, date_format="iso")

stats_df = pd.DataFrame(file_level_stats)
stats_df.to_csv(stats_out_csv, index=False)

constituency_candidate_counts = processed_news.groupby("constituency")["candidates"].count()

print(f"Processed rows: {len(processed_news):,}")
print(f"Unique constituencies: {unique_constituencies:,}")
print(
    "Candidates per constituency -> "
    f"min: {int(constituency_candidate_counts.min())}, "
    f"max: {int(constituency_candidate_counts.max())}, "
    f"mean: {constituency_candidate_counts.mean():.2f}"
)
print(f"Files scanned: {1 + len(dhaka_tribune_referendum_source_files)}")
print(f"Saved: {news_out_csv}")
print(f"Saved: {news_out_json}")
print(f"Saved: {stats_out_csv}")

if {"dataset_type", "source_file"}.issubset(stats_df.columns):
    stats_df.sort_values(["dataset_type", "source_file"]).head(20)
else:
    stats_df.head(20)

Using Dhaka Tribune candidate file: dhaka_tribune_candidates_raw_2026-04-11.csv
Using Dhaka Tribune referendum files: dhaka_tribune_referendum_raw_20260411_012714.csv, dhaka_tribune_seats_raw_2026-04-11.csv
Using Daily Star candidate file: daily_star_candidates_2026-04-11.csv
Processed rows: 2,028
Unique constituencies: 300
Candidates per constituency -> min: 2, max: 15, mean: 6.76
Files scanned: 3
Saved: /home/billy/X/Election-Dashboard/data_science/data/processed/news_scrape/news_scrape_processed.csv
Saved: /home/billy/X/Election-Dashboard/data_science/data/processed/news_scrape/news_scrape_processed.json
Saved: /home/billy/X/Election-Dashboard/data_science/data/processed/news_scrape/news_scrape_file_stats.csv


In [3]:
# Enrich processed output with Dhaka Tribune seat area metadata.
referendum_area_lookup = referendum_df[["constituency", "seats_area"]].copy()
referendum_area_lookup["constituency"] = referendum_area_lookup["constituency"].map(normalize_constituency_name)
referendum_area_lookup["seats_area"] = (
    referendum_area_lookup["seats_area"]
    .astype("string")
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
    .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
)
referendum_area_lookup["lookup_non_null_count"] = referendum_area_lookup[["seats_area"]].notna().sum(axis=1)
referendum_area_lookup = referendum_area_lookup.sort_values(
    ["constituency", "lookup_non_null_count"],
    ascending=[True, False],
)
referendum_area_lookup = referendum_area_lookup.groupby("constituency", as_index=False).agg(
    {"seats_area": first_non_null}
)

processed_news = processed_news.merge(
    referendum_area_lookup,
    on="constituency",
    how="left",
    suffixes=("", "_referendum"),
)

if "seats_area_referendum" in processed_news.columns:
    if "seats_area" in processed_news.columns:
        processed_news["seats_area"] = processed_news["seats_area"].combine_first(
            processed_news["seats_area_referendum"]
        )
    else:
        processed_news["seats_area"] = processed_news["seats_area_referendum"]
    processed_news = processed_news.drop(columns=["seats_area_referendum"])

output_columns = [
    "constituency",
    "candidates",
    "party",
    "alliance",
    "votes",
    "turnout",
    "margin",
    "referendum_yes",
    "referendum_no",
    "male_voter",
    "female_voter",
    "seats_area",
]
processed_news = processed_news[[column for column in output_columns if column in processed_news.columns]]
processed_news = processed_news.sort_values(["constituency", "votes"], ascending=[True, False], na_position="last").reset_index(drop=True)

processed_news.to_csv(news_out_csv, index=False)
processed_news.to_csv(news_out_csv_versioned, index=False)
processed_news.to_json(news_out_json, orient="records", force_ascii=False, indent=2, date_format="iso")
processed_news.to_json(news_out_json_versioned, orient="records", force_ascii=False, indent=2, date_format="iso")

print("Updated output with seats_area and alliance")
print(f"Saved: {news_out_csv}")
print(f"Saved: {news_out_json}")

Updated output with seats_area and alliance
Saved: /home/billy/X/Election-Dashboard/data_science/data/processed/news_scrape/news_scrape_processed.csv
Saved: /home/billy/X/Election-Dashboard/data_science/data/processed/news_scrape/news_scrape_processed.json


In [4]:
processed_news.sample(min(10, len(processed_news)), random_state=42) if len(processed_news) else processed_news.head()

,constituency,candidates,party,votes,turnout,margin,referendum_yes,referendum_no,male_voter,female_voter,seats_area
1284,Moulvibazar 4,Pritam Das,National Citizen Party (NCP),71.0,487892.0,119569.0,<NA>,<NA>,246202.0,241688.0,"Kamolganj, Sreemangal"
982,Jhenaidah 3,Mohammad Mehdi Hasan,BNP,149821.0,431015.0,26036.0,198885.0,111917.0,217470.0,213542.0,"Maheshpur, Kotchandpur"
1544,Netrakona 2,Md. Anwarul Haque,BNP,171399.0,502438.0,104032.0,153793.0,79752.0,252379.0,250047.0,"Netrakona Sadar, Barhatta"
593,Dhaka 4,Syed Md. Mosaddeq Billah,Islami Andolon Bangladesh,6518.0,362506.0,2920.0,<NA>,<NA>,186467.0,176034.0,"Dhaka South City Corporation ( 47, 51, 52, 53,..."
1292,Munshiganj 1,Rokeya Akter,Insaniyat Biplob Bangladesh,320.0,545519.0,65994.0,210728.0,77561.0,280716.0,264802.0,"Srinagar, Sirajdikhan"
781,Gaibandha 5,Shamim Haider Patowary,Jatiya Party (JaPa),3102.0,385261.0,37853.0,<NA>,<NA>,192277.0,192981.0,"Saghata, Phulchari"
1578,Nilphamari 3,Alhaj Syed Ali,BNP,89807.0,294446.0,19879.0,<NA>,<NA>,149878.0,144567.0,Jaldhaka
367,Cox's Bazar 2,Md. Mahmudul Karim,Jatiya Party (JaPa),757.0,387851.0,33654.0,130373.0,87410.0,206453.0,181398.0,"Kutubdia, Maheshkhali"
1702,Pirojpur 1,Alamgir Hossain,BNP,107105.0,392178.0,25554.0,<NA>,<NA>,198709.0,193469.0,"Nazirpur, Nesharabad, Pirojpur Sadar"
1790,Rangpur 4,Abu Sahma,Bangladesh Khelafat Majlish,NaN,509905.0,8331.0,<NA>,<NA>,250787.0,259113.0,"Kaunia, Pirgacha"
